## 🔧 MesoNet Deepfake Detection

---

### 🧪 MesoNet

* MesoNet is a **lightweight Convolutional Neural Network (CNN)** designed for **deepfake detection**.
* It targets the **mesoscopic level** of video frames: not raw pixels, not semantic meaning, but features like compression artifacts, facial textures, and local inconsistencies.
* Ideal for **frame-level classification**, especially for resource-constrained environments.

---

### ❓ Why Use MesoNet?

* Deepfakes often leave subtle inconsistencies that are **not easily seen by humans** but detectable at the CNN level.
* MesoNet is small, fast, and effective for binary classification: **"Real" vs "Fake"**.
* It requires fewer parameters than large models like Xception, making it suitable for mobile/web apps.

---

### 🔹 MesoNet Architecture Overview

This implementation follows the **original Meso-4 architecture**, which consists of:

#### 🏠 Layers Breakdown:

1. **Convolutional Block 1:**

   * Conv2D with 8 filters (3x3)
   * Batch Normalization
   * ReLU Activation
   * Max Pooling (2x2)

2. **Convolutional Block 2:**

   * Conv2D with 8 filters (5x5)
   * Batch Normalization
   * ReLU Activation
   * Max Pooling (2x2)

3. **Convolutional Block 3:**

   * Conv2D with 16 filters (5x5)
   * Batch Normalization
   * ReLU Activation
   * Max Pooling (2x2)

4. **Convolutional Block 4:**

   * Conv2D with 16 filters (5x5)
   * Batch Normalization
   * ReLU Activation
   * Max Pooling (4x4)

5. **Fully Connected Head:**

   * Flatten layer
   * Dense layer with 16 neurons + ReLU
   * Dropout (0.5)
   * Output Dense layer with sigmoid activation (binary output: real or fake)

---

### 📚 How Did We Train MesoNet?

#### 1. **Dataset Structure**

* The dataset includes two folders: `real` and `fake`, each with videos.
* Extracted frames are stored inside `/frames/[video_id]/[frame].png`.

#### 2. **Data Preparation**

* Custom `FrameGenerator` was created to load batches from image paths.
* Each image is resized to **256x256**, normalized to \[0, 1], and batched.

#### 3. **Training Strategy**

* The model is compiled with:

  * Loss: `binary_crossentropy`
  * Optimizer: `Adam`
  * Metric: `accuracy`
* Training uses up to **20 epochs** with **early stopping** based on:

  * Accuracy ≥ 80%
  * Precision ≥ 80%
  * F1-score ≥ 80%
* After training, the model is saved as `mesonet_model.h5`

#### 4. **Evaluation**

* A custom callback (`MetricsCallback`) calculates accuracy, precision, recall, and F1 after every epoch using the test set.
* Early stopping prevents overfitting and speeds up convergence.

#### 5. **Prediction Function**

* Any new image (frame) can be resized, normalized, and passed into the model.
* The model outputs a confidence score:

  * **Closer to 0** → Real
  * **Closer to 1** → Fake

---

### 🔹 Conclusion

* MesoNet is optimized for frame-based classification of faces.
* Its simplicity and efficiency make it suitable for deployment in web/mobile apps.
* The training pipeline ensures high accuracy without using heavy pre-trained models.




In [ ]:
# STEP 0: INSTALL DEPENDENCY
# Installs the tqdm package for showing progress bars during training/validation.
!pip install tqdm

# STEP 1: MOUNT GOOGLE DRIVE
# Mounts your Google Drive to Colab so the dataset and model can be accessed and saved.
from google.colab import drive
drive.mount('/content/drive')

# STEP 2: IMPORT LIBRARIES
# This imports:
"""
cv2 (OpenCV) for image processing
numpy for array operations
glob for file path matching
tensorflow for defining and training the MesoNet model
sklearn.metrics for precision, recall, F1-score
tqdm for progress bars
"""
import os, cv2, numpy as np
from glob import glob
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input, BatchNormalization, Activation

# STEP 3: BUILD DATASET INDEX (IMAGE PATH + LABEL)
"""
Crawls through the real/frames/ and fake/frames/ directories
Collects paths to all .png images
Tags them with a label (0 for real, 1 for fake)
"""
def build_dataset_list(base_path, label):
    video_dirs = glob(os.path.join(base_path, '*'))
    samples = []
    for v in video_dirs:
        frames = glob(os.path.join(v, '*.png')) + glob(os.path.join(v, '*.PNG'))
        for f in frames:
            samples.append((f, label))
    return samples

real_path = '/content/drive/MyDrive/DataSet/real/frames'
fake_path = '/content/drive/MyDrive/DataSet/fake/frames'

real_samples = build_dataset_list(real_path, 0)
fake_samples = build_dataset_list(fake_path, 1)
all_samples = real_samples + fake_samples

train_samples, test_samples = train_test_split(
    all_samples, test_size=0.2,
    stratify=[s[1] for s in all_samples],
    random_state=42
)

# STEP 4: SAFE FRAME GENERATOR
"""
Used to load images in batches while training/testing.
Reads image paths and labels
Resizes each image to 256x256
Normalizes pixel values [0-1]
Returns NumPy arrays in batches
Shuffles at the end of each epoch (for training only)
"""

class FrameGenerator(Sequence):
    def __init__(self, samples, batch_size=32, shuffle=True):
        self.samples = samples
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.samples) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.samples[idx * self.batch_size:(idx + 1) * self.batch_size]
        images, labels = [], []
        for path, label in batch:
            img = cv2.imread(path)
            if img is None:
                continue
            img = cv2.resize(img, (256, 256))
            img = img / 255.0
            images.append(img)
            labels.append(label)
        if len(images) == 0:
            return np.zeros((1, 256, 256, 3)), np.zeros((1,))
        return np.array(images), np.array(labels)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.samples)

train_gen = FrameGenerator(train_samples)
test_gen = FrameGenerator(test_samples, shuffle=False)

# STEP 5: BUILD MESONET
"""
This builds the original Meso-4 architecture:
Layer-by-Layer Breakdown:
Conv2D (8 filters, 3x3)  → BatchNorm → ReLU → MaxPool(2x2)
Conv2D (8 filters, 5x5)  → BatchNorm → ReLU → MaxPool(2x2)
Conv2D (16 filters, 5x5) → BatchNorm → ReLU → MaxPool(2x2)
Conv2D (16 filters, 5x5) → BatchNorm → ReLU → MaxPool(4x4)
Flatten → Dense(16) → Dropout(0.5) → Dense(1, sigmoid)
Output:
A probability (0 = real, 1 = fake)
Used to detect deepfake frames
"""

def build_meso():
    inp = Input(shape=(256, 256, 3))
    x = Conv2D(8, (3,3), padding='same')(inp)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2,2))(x)

    x = Conv2D(8, (5,5), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2,2))(x)

    x = Conv2D(16, (5,5), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(2,2))(x)

    x = Conv2D(16, (5,5), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(pool_size=(4,4))(x)

    x = Flatten()(x)
    x = Dense(16, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(1, activation='sigmoid')(x)

    return Model(inputs=inp, outputs=out)

model = build_meso()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# STEP 6: METRICS CALLBACK WITH tqdm AND EARLY STOP
"""
Adds custom logic after each epoch:
Loops through the test set
Calculates:
Accuracy
Precision
Recall
F1-score
Stops training early if all metrics ≥ 80%
"""
class MetricsCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []
        max_batches = len(test_gen)

        print(f"\n🔎 Evaluating validation set (Epoch {epoch+1})...")

        for i, (x_batch, y_batch) in enumerate(tqdm(test_gen, total=max_batches, desc="Validating")):
            if i >= max_batches:
                break
            if len(x_batch) == 0:
                continue
            preds = (model.predict(x_batch, verbose=0) > 0.5).astype("int32")
            y_true.extend(y_batch)
            y_pred.extend(preds.flatten())

        if len(y_true) == 0:
            print("⚠️ Skipped metric calculation (no valid test data)")
            return

        acc = logs.get('accuracy', 0)
        prec = precision_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        print(f"✅ Epoch {epoch+1} — Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
        if acc >= 0.80 and prec >= 0.80 and f1 >= 0.80:
            print("🎉 Early stopping: all metrics ≥ 80%")
            self.model.stop_training = True

# STEP 7: TRAIN THE MODEL
"""
Starts training the model:
Up to 20 epochs
Each epoch = full pass through the dataset
Uses train_gen and test_gen to load batches
Evaluation happens at the end of each epoch
"""

model.fit(train_gen, epochs=20, validation_data=test_gen, callbacks=[MetricsCallback()])

# STEP 8: SAVE TRAINED MODEL
# Saves the trained model in .h5 format so you can reuse it later or deploy it.
model.save('/content/mesonet_model.h5')
print("✅ Model saved at /content/mesonet_model.h5")

# STEP 9: PREDICT ANY IMAGE FRAME
"""
Takes path to an image
Resizes it to 256x256
Normalizes pixel values
Predicts using the trained model
Prints label (Real or Fake) and confidence score
"""
def predict_frame(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (256, 256))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)
    pred = model.predict(img)[0][0]
    label = "Fake" if pred > 0.5 else "Real"
    print(f"🧠 Prediction: {label} (Confidence: {pred:.4f})")


In [ ]:
!cp /content/mesonet_model.h5 /content/drive/MyDrive/mesonet_model.h5


Testing Model


In [ ]:
predict_frame('/content/drive/MyDrive/DataSet/fake/frames/0000_fake/028.png')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
🧠 Prediction: Fake (Confidence: 0.9919)


In [ ]:
predict_frame('/content/drive/MyDrive/DataSet/real/frames/0000/000.png')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
🧠 Prediction: Real (Confidence: 0.2384)


In [ ]:
predict_frame('/content/drive/MyDrive/DataSet/real/frames/0000/009.png')


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
🧠 Prediction: Real (Confidence: 0.1040)


In [ ]:
from google.colab import files
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from PIL import Image
import io

# ✅ Load trained model (if not already loaded)
model = load_model('/content/mesonet_model.h5')

# ✅ Upload image
uploaded = files.upload()

# ✅ Predict function
def predict_uploaded_image(file_path):
    # Read image using OpenCV
    img = cv2.imread(file_path)

    if img is None:
        print("❌ Could not read the image.")
        return

    # Resize to 256x256 if not already
    if img.shape[:2] != (256, 256):
        img = cv2.resize(img, (256, 256))

    # Normalize and reshape
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    # Predict
    pred = model.predict(img)[0][0]
    label = "Fake" if pred > 0.5 else "Real"
    confidence_fake = pred
    confidence_real = 1 - pred

    print(f"🧠 Prediction: {label}")
    print(f"→ Confidence Fake: {confidence_fake:.4f}")
    print(f"→ Confidence Real: {confidence_real:.4f}")

# ✅ Run prediction on uploaded image(s)
for filename in uploaded.keys():
    print(f"\n📂 Processing: {filename}")
    predict_uploaded_image(filename)


Saving fake4.png to fake4.png

📂 Processing: fake4.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step
🧠 Prediction: Real
→ Confidence Fake: 0.2443
→ Confidence Real: 0.7557


Create Folder Structure in Colab

In [ ]:
# Create the hamster library folder
!mkdir -p hamster
!touch hamster/__init__.py


Move Trained Model to Library Folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/mesonet_model.h5 hamster/


Mounted at /content/drive


Create predictor.py Inside hamster/

In [ ]:
%%writefile hamster/predictor.py
import os
import cv2
import numpy as np
from tensorflow.keras.models import load_model

class HamsterDetector:
    def __init__(self, model_path=None):
        if model_path is None:
            model_path = os.path.join(os.path.dirname(__file__), 'mesonet_model.h5')
        self.model = load_model(model_path)

    def predict(self, image_path):
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError("Image could not be read.")
        img = cv2.resize(img, (256, 256))
        img = img / 255.0
        img = np.expand_dims(img, axis=0)
        pred = self.model.predict(img)[0][0]
        label = "Fake" if pred > 0.5 else "Real"
        return {
            "prediction": label,
            "confidence_fake": round(float(pred), 4),
            "confidence_real": round(1.0 - float(pred), 4)
        }


Writing hamster/predictor.py


Test Script

In [ ]:
from hamster.predictor import HamsterDetector

detector = HamsterDetector()
result = detector.predict('/content/drive/MyDrive/DataSet/fake/frames/0000_fake/065.png')
print(result)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 447ms/step
{'prediction': 'Fake', 'confidence_fake': 0.9931, 'confidence_real': 0.0069}


Create requirements.txt

In [ ]:
%%writefile hamster/requirements.txt
tensorflow>=2.8
opencv-python
numpy


Writing hamster/requirements.txt


Zip the Folder

In [ ]:
!zip -r hamster.zip hamster


  adding: hamster/ (stored 0%)
  adding: hamster/requirements.txt (stored 0%)
  adding: hamster/__init__.py (stored 0%)
  adding: hamster/mesonet_model.h5 (deflated 26%)
  adding: hamster/__pycache__/ (stored 0%)
  adding: hamster/__pycache__/__init__.cpython-311.pyc (deflated 26%)
  adding: hamster/__pycache__/predictor.cpython-311.pyc (deflated 44%)
  adding: hamster/predictor.py (deflated 51%)


In [ ]:
from hamster.predictor import HamsterDetector

detector = HamsterDetector()
result = detector.predict('/content/drive/MyDrive/DataSet/fake/frames/0000_fake/028.png')
print(result)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 888ms/step
{'prediction': 'Fake', 'confidence_fake': 0.9919, 'confidence_real': 0.0081}


In [ ]:
!cp -r hamster /content/drive/MyDrive/


In [ ]:
!cd /content/drive/MyDrive && zip -r hamster.zip hamster


  adding: hamster/ (stored 0%)
  adding: hamster/__init__.py (stored 0%)
  adding: hamster/mesonet_model.h5 (deflated 26%)
  adding: hamster/predictor.py (deflated 51%)
  adding: hamster/__pycache__/ (stored 0%)
  adding: hamster/__pycache__/__init__.cpython-311.pyc (deflated 26%)
  adding: hamster/__pycache__/predictor.cpython-311.pyc (deflated 44%)
  adding: hamster/requirements.txt (stored 0%)


In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/hamster.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>